In [1]:
import scipy
import torch
import numpy as np
from torch import nn
from torchmetrics.functional import pairwise_cosine_similarity

In [5]:
def _batch_mahalanobis(bL, bx):
    r"""
    Computes the squared Mahalanobis distance :math:`\mathbf{x}^\top\mathbf{M}^{-1}\mathbf{x}`
    for a factored :math:`\mathbf{M} = \mathbf{L}\mathbf{L}^\top`.

    Accepts batches for both bL and bx. They are not necessarily assumed to have the same batch
    shape, but `bL` one should be able to broadcasted to `bx` one.
    """
    n = bx.size(-1)
    bx_batch_shape = bx.shape[:-1]

    # Assume that bL.shape = (i, 1, n, n), bx.shape = (..., i, j, n),
    # we are going to make bx have shape (..., 1, j,  i, 1, n) to apply batched tri.solve
    bx_batch_dims = len(bx_batch_shape)
    bL_batch_dims = bL.dim() - 2
    outer_batch_dims = bx_batch_dims - bL_batch_dims
    old_batch_dims = outer_batch_dims + bL_batch_dims
    new_batch_dims = outer_batch_dims + 2 * bL_batch_dims
    # Reshape bx with the shape (..., 1, i, j, 1, n)
    bx_new_shape = bx.shape[:outer_batch_dims]
    for sL, sx in zip(bL.shape[:-2], bx.shape[outer_batch_dims:-1]):
        bx_new_shape += (sx // sL, sL)
    bx_new_shape += (n,)
    bx = bx.reshape(bx_new_shape)
    # Permute bx to make it have shape (..., 1, j, i, 1, n)
    permute_dims = (
        list(range(outer_batch_dims))
        + list(range(outer_batch_dims, new_batch_dims, 2))
        + list(range(outer_batch_dims + 1, new_batch_dims, 2))
        + [new_batch_dims]
    )
    bx = bx.permute(permute_dims)

    flat_L = bL.reshape(-1, n, n)  # shape = b x n x n
    flat_x = bx.reshape(-1, flat_L.size(0), n)  # shape = c x b x n
    flat_x_swap = flat_x.permute(1, 2, 0)  # shape = b x n x c
    M_swap = (
        torch.linalg.solve_triangular(flat_L, flat_x_swap, upper=False).pow(2).sum(-2)
    )  # shape = b x c
    M = M_swap.t()  # shape = c x b

    # Now we revert the above reshape and permute operators.
    permuted_M = M.reshape(bx.shape[:-1])  # shape = (..., 1, j, i, 1)
    permute_inv_dims = list(range(outer_batch_dims))
    for i in range(bL_batch_dims):
        permute_inv_dims += [outer_batch_dims + i, old_batch_dims + i]
    reshaped_M = permuted_M.permute(permute_inv_dims)  # shape = (..., 1, i, j, 1)
    return reshaped_M.reshape(bx_batch_shape)

RuntimeError: The size of tensor a (1200) must match the size of tensor b (32) at non-singleton dimension 0

In [19]:
cost = np.array([
    [1, 5, 5, 5],
    [2, 1, 3, 3],
    # [4, 4, 4, 1]
])
type(scipy.optimize.linear_sum_assignment(cost)[0])

numpy.ndarray

In [12]:
concepts = torch.randn(1200, 128)
prototypes = torch.randn(30, 128)


In [17]:
p_norm = prototypes / prototypes.norm(dim=-1, keepdim=True)
c_norm = concepts / concepts.norm(dim=-1, keepdim=True)
p_norm @ c_norm.T

tensor([[ 0.0212, -0.0756, -0.1675,  ..., -0.0792,  0.1766, -0.0286],
        [-0.0732,  0.0120, -0.0105,  ..., -0.0249, -0.0750,  0.0005],
        [-0.1075, -0.2058, -0.0618,  ..., -0.0250, -0.0023,  0.0226],
        ...,
        [-0.1232,  0.0116, -0.0358,  ...,  0.0799,  0.2687, -0.1086],
        [-0.0377, -0.0171, -0.0322,  ..., -0.0385, -0.0175,  0.0442],
        [ 0.0079, -0.0116,  0.0648,  ...,  0.1709, -0.0636,  0.0895]])

In [18]:
pairwise_cosine_similarity(prototypes, concepts)

tensor([[ 0.0212, -0.0756, -0.1675,  ..., -0.0792,  0.1766, -0.0286],
        [-0.0732,  0.0120, -0.0105,  ..., -0.0249, -0.0750,  0.0005],
        [-0.1075, -0.2058, -0.0618,  ..., -0.0250, -0.0023,  0.0226],
        ...,
        [-0.1232,  0.0116, -0.0358,  ...,  0.0799,  0.2687, -0.1086],
        [-0.0377, -0.0171, -0.0322,  ..., -0.0385, -0.0175,  0.0442],
        [ 0.0079, -0.0116,  0.0648,  ...,  0.1709, -0.0636,  0.0895]])